# 02 — Random Forest: datasets y modelo base

Este notebook carga **Contact Lenses (discreto), Iris (continuo) y Heart Disease
(mixto)**, prepara una partición estratificada reproducible y entrena un Random Forest
para cada uno. Las definiciones y fuentes completas están en el [README](../README.md).

**Contrato compartido:** `predict(X)` recibe un DataFrame con las características y
retorna un DataFrame con una única columna `prediccion`, conservando el índice y orden
de `X`. `X` e `y` se conservan como DataFrames; `y` tiene una columna `objetivo`.
El entrenamiento usa directamente scikit-learn, sin una clase propia.

Ejecutar todas las celdas en orden con el entorno del proyecto. Los datos de UCI están
incluidos en `data/raw/`; Iris viene con scikit-learn. La carga no utiliza Internet.

In [ ]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
# Random Forest viene de scikit-learn, como permite la consigna; el árbol es propio.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.validation import check_is_fitted

# Compartir la carga y las mismas particiones sin ejecutar el otro notebook.
# Busca la raíz para poder ejecutar el notebook desde cualquier carpeta del repositorio.
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos_tp.py").is_file() and (carpeta / "data/raw").is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError("Ejecutá desde la raíz del repositorio o notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from datos_tp import (
    SEMILLA, FRACCION_TEST, TAMANOS, CLASES, cargar_datasets, particionar,
)

print(f"Python {platform.python_version()} | pandas {pd.__version__} | "
      f"NumPy {np.__version__} | scikit-learn {sklearn.__version__}")

## 1. Carga y preparación de los datos

Los archivos originales se comprueban con SHA-256 para detectar cambios accidentales.
La preparación interpreta sus códigos según las descripciones originales, excluye
identificadores de las características y comprueba que el objetivo no tenga faltantes.
No se eliminan filas: Heart Disease tiene 6 casos con un atributo faltante, que RF
completa dentro de su pipeline.

Contact Lenses tiene **cuatro** atributos: se incluye la producción de lágrimas,
según `lenses.names`. En Heart, las categorías codificadas con números se convierten
a texto para distinguirlas de las medidas numéricas.

La implementación de la carga está en `datos_tp.py`, compartida con C4.5.


In [ ]:
datasets = cargar_datasets()

# Comprueba que cada dataset tenga el tamaño y las clases esperadas antes de usarlo.
# Así un archivo modificado por error se detecta acá y no en un resultado raro.
resumen_datos = []
for nombre, datos in datasets.items():
    X, y = datos["X"], datos["y"]
    assert X.shape == TAMANOS[nombre]
    assert y.shape == (len(X), 1) and list(y.columns) == ["objetivo"]
    assert X.index.is_unique and X.index.equals(y.index)
    assert not y.isna().any().any()
    assert not np.isinf(X.select_dtypes(include="number").to_numpy()).any()
    assert y["objetivo"].value_counts().to_dict() == CLASES[nombre]
    resumen_datos.append({
        "dataset": nombre, "tipo": datos["tipo"], "casos": len(X),
        "atributos": X.shape[1], "clases": y["objetivo"].nunique(),
        "numericos": X.select_dtypes(include="number").shape[1],
        "categoricos": X.select_dtypes(exclude="number").shape[1],
    })

display(pd.DataFrame(resumen_datos).set_index("dataset"))

## 2. Partición estratificada

Se reserva el 20 % para prueba, con semilla 42. `stratify` mantiene aproximadamente
las proporciones de las clases. Los índices originales se conservan para poder usar
**exactamente los mismos casos** en C4.5. La cantidad de filas debe ser entera: en
Contact Lenses quedan 19 casos de entrenamiento y 5 de prueba.

In [ ]:
# Una partición por dataset, con la misma semilla que usa el árbol propio.
particiones = {
    nombre: particionar(datos["X"], datos["y"])
    for nombre, datos in datasets.items()
}

distribuciones = []
for nombre, parte in particiones.items():
    X_train, X_test = parte["X_train"], parte["X_test"]
    y_train, y_test = parte["y_train"], parte["y_test"]
    assert X_train.index.intersection(X_test.index).empty
    assert len(X_train) + len(X_test) == len(datasets[nombre]["X"])
    assert X_train.index.equals(y_train.index)
    assert X_test.index.equals(y_test.index)
    assert set(y_train["objetivo"]) == set(y_test["objetivo"]) == set(CLASES[nombre])
    for clase, total in CLASES[nombre].items():
        distribuciones.append({
            "dataset": nombre, "clase": clase, "total": total,
            "train": int(y_train["objetivo"].eq(clase).sum()),
            "test": int(y_test["objetivo"].eq(clase).sum()),
        })

display(pd.DataFrame(distribuciones).set_index(["dataset", "clase"]))

## 3. Modelo base y adaptación de `predict`

El pipeline transforma las categorías mediante one-hot y conserva las columnas
numéricas. El codificador aprende las categorías **solo en entrenamiento**. Una
categoría no observada en entrenamiento se representa con ceros en su grupo de
columnas (`handle_unknown="ignore"`). No se escalan las medidas numéricas.

Los faltantes se completan dentro del pipeline: mediana en los atributos numéricos y
valor más frecuente en los categóricos. Se completan en lugar de descartar los casos
para no perder información y para que C4.5 y RF usen las mismas particiones. Los
reemplazos se calculan **solo con entrenamiento**, sin información del test.

El Random Forest usa por defecto 100 árboles, Gini, bootstrap, `max_features="sqrt"`,
sin límite de profundidad y semilla 42. `crear_modelo` permite variar
`n_estimators`, `max_features`, `max_depth`, `criterion`, `bootstrap` y
`random_state`; cada llamada crea un pipeline nuevo. Se fija `n_jobs=1` para
comparar tiempos con el mismo grado de paralelismo. La búsqueda de configuraciones
se realiza en el notebook 03.

`crear_predictor` recibe el pipeline ya entrenado y devuelve una función `predict(X)`
asociada a ese modelo. La función verifica la entrada y convierte el array de
scikit-learn al DataFrame del contrato. No exige una clase nueva ni un entrenamiento
común con C4.5.

Referencias: [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
[OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
y [SimpleImputer](https://scikit-learn.org/stable/modules/impute.html).

In [ ]:
# Arma el flujo completo: completar faltantes, codificar categorías y entrenar.
# Por qué un Pipeline: todo se aprende solo con entrenamiento y no se filtra
# información del conjunto de prueba.
def crear_modelo(X_train, *, n_estimators=100, max_features="sqrt",
                max_depth=None, criterion="gini", bootstrap=True,
                random_state=SEMILLA):
    """Crea un RF nuevo con preprocesamiento ajustable solo en entrenamiento."""
    categoricas = X_train.select_dtypes(exclude="number").columns.tolist()
    numericas = X_train.select_dtypes(include="number").columns.tolist()
    preprocesador = ColumnTransformer([
        # Categorías: valor más frecuente para los faltantes y one-hot, porque scikit-learn solo
        # acepta números. Números: mediana, que no se deja arrastrar por valores extremos.
        ("categoricas", Pipeline([
            ("imputador", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categoricas),
        ("numericas", SimpleImputer(strategy="median"), numericas),
    ])
    return Pipeline([
        ("preprocesador", preprocesador),
        ("random_forest", RandomForestClassifier(
            n_estimators=n_estimators, criterion=criterion, max_depth=max_depth,
            min_samples_split=2, min_samples_leaf=1, max_features=max_features,
            bootstrap=bootstrap, random_state=random_state, n_jobs=1
        )),
    ])


# Adapta el modelo al contrato común: DataFrame con la columna prediccion y el mismo
# índice de entrada. Así C4.5 y Random Forest se evalúan con el mismo código.
def crear_predictor(modelo):
    """Asocia un pipeline entrenado a una función con el contrato predict(X)."""
    check_is_fitted(modelo.named_steps["random_forest"])
    columnas = pd.Index(modelo.feature_names_in_)

    def predict(X: pd.DataFrame) -> pd.DataFrame:
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un pandas.DataFrame.")
        if not X.columns.equals(columnas):
            raise ValueError("X debe tener las mismas columnas y orden que en entrenamiento.")
        if X.empty:
            raise ValueError("X debe contener casos.")
        return pd.DataFrame({"prediccion": modelo.predict(X)}, index=X.index.copy())

    return predict

## 4. Entrenar y predecir en los tres datasets

Se conserva un pipeline entrenado y una función de predicción por dataset.
`y_train` sigue siendo un DataFrame; se pasa su columna `objetivo` a scikit-learn,
que espera un vector de etiquetas para esta clasificación.

Las tablas muestran casos de prueba, sus respuestas reales y las predicciones.
La accuracy es la proporción de casos acertados y sirve como resultado base de esta
prueba; todavía no es una comparación con C4.5.

In [ ]:
# Entrena un bosque por dataset y guarda sus predicciones para las comprobaciones.
modelos = {}
predictores = {}
predicciones = {}
resultados = []

for nombre, parte in particiones.items():
    modelo = crear_modelo(parte["X_train"])
    modelo.fit(parte["X_train"], parte["y_train"]["objetivo"])
    predict = crear_predictor(modelo)
    salida = predict(parte["X_test"])

    modelos[nombre] = modelo
    predictores[nombre] = predict
    predicciones[nombre] = salida
    resultados.append({
        "dataset": nombre, "train": len(parte["X_train"]),
        "test": len(parte["X_test"]),
        "accuracy": accuracy_score(parte["y_test"]["objetivo"], salida["prediccion"]),
    })
    print(f"{nombre}: primeras predicciones de prueba")
    display(parte["y_test"].join(salida).head())

resultados = pd.DataFrame(resultados).set_index("dataset")
display(resultados)

## 5. Verificación del contrato y de reproducibilidad

Se comprueba que la salida conserve índices, orden, cantidad de filas y etiquetas,
y que predecir no modifique la entrada. Se repiten la carga, la partición y el
entrenamiento para comprobar igualdad de datos y resultados con la misma semilla.
También se verifica que el codificador haya aprendido solo las categorías de train.

Estas comprobaciones detectan errores de integración. No exigen un umbral de accuracy:
con cinco casos de prueba en Contact Lenses, un solo error cambia el resultado en
20 puntos porcentuales. No se debe ajustar el modelo mirando este test.

In [ ]:
# Comprueba que repetir todo dé igual resultado y que el contrato se cumpla, incluso
# con los casos en orden invertido.
datos_repetidos = cargar_datasets()
for nombre, parte in particiones.items():
    pd.testing.assert_frame_equal(datasets[nombre]["X"], datos_repetidos[nombre]["X"])
    pd.testing.assert_frame_equal(datasets[nombre]["y"], datos_repetidos[nombre]["y"])

    # Un orden invertido comprueba que predict no ordene ni reinicie los índices.
    entrada = parte["X_test"].iloc[::-1].copy(deep=True)
    original = entrada.copy(deep=True)
    salida = predictores[nombre](entrada)
    assert isinstance(salida, pd.DataFrame)
    assert salida.shape == (len(entrada), 1)
    assert list(salida.columns) == ["prediccion"]
    assert salida.index.equals(entrada.index)
    assert not salida.isna().any().any()
    assert set(salida["prediccion"]) <= set(CLASES[nombre])
    pd.testing.assert_frame_equal(entrada, original)
    pd.testing.assert_frame_equal(salida, predicciones[nombre].iloc[::-1])

    repetida = particionar(datos_repetidos[nombre]["X"], datos_repetidos[nombre]["y"])
    for clave in parte:
        pd.testing.assert_frame_equal(parte[clave], repetida[clave])

    categoricas = parte["X_train"].select_dtypes(exclude="number").columns
    if len(categoricas):
        preprocesador = modelos[nombre].named_steps["preprocesador"]
        encoder = preprocesador.named_transformers_["categoricas"].named_steps["one_hot"]
        for columna, categorias in zip(categoricas, encoder.categories_):
            assert set(categorias) == set(parte["X_train"][columna].dropna())

    modelo_repetido = crear_modelo(repetida["X_train"])
    modelo_repetido.fit(repetida["X_train"], repetida["y_train"]["objetivo"])
    pd.testing.assert_frame_equal(
        predicciones[nombre], crear_predictor(modelo_repetido)(repetida["X_test"])
    )
    print(f"OK — {nombre}: carga, partición, contrato y predicciones reproducibles.")

## Uso del contrato para comparar después

Seleccionar el predictor entrenado del dataset y pasarle sus casos de prueba.
El notebook 01 implementa el mismo contrato para C4.5 categórico y numérico.
Ambos usan `datos_tp.py` para obtener las mismas entradas y particiones.
Comparar con `y_test` respetando los índices; las respuestas nunca entran en `X`.

La comparación experimental con distintas parametrizaciones se desarrolla
en el notebook 03. Estas ejecuciones base no seleccionan parámetros usando test.


In [ ]:
# Ejemplo de uso: predice con un modelo ya entrenado y lo compara con la clase real.
predict = predictores["iris"]
X_test = particiones["iris"]["X_test"]
y_test = particiones["iris"]["y_test"]
display(y_test.join(predict(X_test)).head())